# 1.2 - Pembuatan Databricks Vector Search & Strategi Retrieval

Notebook ini merupakan tahap inti dalam implementasi **Retrieval-Augmented Generation (RAG)** di Databricks. Di sini kita membuat **Vector Search Endpoint**, mengonfigurasi **Delta Sync Vector Search Index** terkelola pada tabel lapisan Gold (`gold_docs_chunked`), serta mengeksplorasi berbagai teknik pencarian (*retrieval*) dan perangkingan ulang (*reranking*).

### Ringkasan Proses:
1. **Instalasi Pustaka Pendukung**: Menginstal `databricks-vectorsearch` dan `flashrank` lalu merestart sesi Python.
2. **Setup Lingkungan Lab**: Menginisialisasi katalog, skema, volume, dan variabel konfigurasi dari `../Includes/Lab_Setup`.
3. **Inisialisasi Vector Search Endpoint**: Memastikan endpoint komputasi Vector Search (`vs_endpoint`) berstatus aktif dan siap melayani query.
4. **Aktivasi Change Data Feed (CDF)**: Mengaktifkan CDF pada tabel Delta sumber agar Vector Search dapat menyinkronkan data secara inkremental.
5. **Pembuatan Managed Vector Search Index**: Membuat indeks vektor terkelola dengan model embedding `databricks-gte-large-en` dan pipeline bertipe `TRIGGERED`.
6. **Eksplorasi Strategi Retrieval**:
   - **Dense Similarity Search**: Pencarian kesamaan makna berbasis representasi vektor semantik.
   - **Full-Text Search**: Pencarian kata kunci berbasis kecocokan leksikal (*keyword matching*).
   - **Hybrid Search**: Penggabungan pencarian semantik dan leksikal untuk akurasi maksimal.
   - **Filtered Search**: Pencarian semantik dengan filter metadata (misalnya mengecualikan dokumen tertentu).
7. **Eksplorasi Strategi Re-ranking**:
   - **Databricks Managed Reranker**: Perangkingan ulang hasil retrieval menggunakan modul reranker Databricks.
   - **FlashRank Local Reranker**: Perangkingan ulang berkecepatan tinggi menggunakan model cross-encoder lokal (`rank-T5-flan`).


### Langkah 1: Instalasi Library Pendukung
Menginstal pustaka resmi `databricks-vectorsearch` untuk berinteraksi dengan layanan Databricks Vector Search dan `flashrank` untuk pemeringkatan ulang (*reranking*) lokal, lalu merestart sesi Python agar modul dapat langsung diakses.


In [0]:
# Menginstal pustaka Databricks Vector Search SDK dan FlashRank untuk reranking
%pip install --quiet --upgrade databricks-vectorsearch flashrank
# Merestart kernel Python agar pustaka yang baru diinstal dapat langsung digunakan
%restart_python

### Langkah 2: Setup Lingkungan Lab
Menjalankan notebook setup untuk menginisialisasi variabel konfigurasi lab (seperti objek `course`, katalog, skema, dan volume di Unity Catalog).


In [0]:
%run ../Includes/Lab_Setup

### Langkah 3: Inisialisasi Vector Search Client & Endpoint
Menyiapkan endpoint komputasi Vector Search (`vs_endpoint`) bertipe `STANDARD`. Endpoint ini berfungsi sebagai cluster terkelola yang melayani komputasi pencarian indeks vektor.


In [0]:
from databricks.vector_search.client import VectorSearchClient

# Menginisialisasi client Vector Search Databricks
vs_client = VectorSearchClient(disable_notice=True)
vs_endpoint_name = "vs_endpoint"

# Mengecek ketersediaan endpoint Vector Search; jika sudah ada tunggu hingga siap (READY), jika belum buat baru
if vs_client.endpoint_exists(vs_endpoint_name):
  vs_client.wait_for_endpoint(vs_endpoint_name)
else:
  vs_client.create_endpoint_and_wait(vs_endpoint_name, endpoint_type='STANDARD')

### Langkah 4: Mengaktifkan Delta Change Data Feed (CDF)
Mengaktifkan fitur Change Data Feed (`delta.enableChangeDataFeed = true`) pada tabel Delta `gold_docs_chunked`. Fitur ini wajib diaktifkan agar Vector Search dapat mendeteksi perubahan data (insert, update, delete) dan menyinkronkan embedding secara otomatis.


In [0]:
# Menentukan nama lengkap tabel Delta sumber pada lapisan Gold (catalog.schema.table)
source_table_fullname = f"{course.catalog}.{course.schema}.gold_docs_chunked"

# Mengaktifkan Change Data Feed (CDF) agar Vector Search dapat menyinkronkan data secara inkremental
spark.sql(f"ALTER TABLE {source_table_fullname} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

### Langkah 5: Membuat Delta Sync Vector Search Index (Managed Embeddings)
Membuat indeks vektor terkelola (*Managed Index*) bernama `dphone_vs_index`. Databricks akan secara otomatis menghitung embedding dari kolom `chunk` menggunakan model embedding terkelola `databricks-gte-large-en` dan memperbarui indeks saat disinkronkan.


In [0]:
# Menentukan nama lengkap index di Unity Catalog (catalog.schema.index_name)
vs_index_fullname = f"{course.catalog}.{course.schema}.dphone_vs_index"

# Jika index belum ada, buat index baru dengan sinkronisasi Delta Sync dan embedding terkelola
if not vs_client.index_exists(vs_endpoint_name, vs_index_fullname):
  vs_index = vs_client.create_delta_sync_index_and_wait(
    source_table_name=source_table_fullname,                  # Tabel Delta sumber di lapisan Gold
    index_name=vs_index_fullname,                             # Nama index vektor di Unity Catalog
    primary_key="id",                                         # Kolom primary key unik
    embedding_source_column="chunk",                          # Kolom teks sumber untuk dihitung embedding-nya
    embedding_model_endpoint_name="databricks-gte-large-en",  # Model embedding terkelola bawaan Databricks
    endpoint_name=vs_endpoint_name,                           # Endpoint Vector Search tujuan
    pipeline_type="TRIGGERED",                                # Mode sinkronisasi: TRIGGERED atau STREAMING
  )
else:
  # Jika index sudah ada, ambil objek index dan tunggu hingga proses sinkronisasi selesai
  vs_index = vs_client.get_index(vs_endpoint_name, vs_index_fullname)
  vs_index.wait_until_ready(wait_for_updates=True)

### Langkah 6: Strategi Retrieval 1 - Pencarian Semantik (Dense Vector Search)
Menjalankan pencarian kesamaan semantik (*semantic similarity search*) menggunakan vektor padat (*dense vectors*). Metode ini memahami makna kontekstual dari query meskipun kata yang digunakan tidak sama persis dengan yang ada di dokumen.


In [0]:
# Query pertanyaan pengguna mengenai fitur produk
query_1 = "Explain the features of the dPhone D1 Lite"

# Menjalankan similarity search untuk mencari 3 potongan dokumen paling relevan secara semantik
results = vs_index.similarity_search(
    query_text=query_1,
    columns=["path", "chunk"],
    num_results=3,
)

# Mengekstrak list hasil dokumen dari response Vector Search
passages = results.get("result", {}).get("data_array", [])

# Menampilkan potongan teks dan path dokumen yang ditemukan
display(passages)

### Langkah 7: Strategi Retrieval 2 - Pencarian Kata Kunci (Full-Text Search)
Menjalankan pencarian berbasis kata kunci (*sparse/keyword search*) dengan parameter `query_type="FULL_TEXT"`. Metode ini sangat efektif untuk mencocokkan istilah spesifik, nomor model, atau akronim (seperti `"6G broadband"`).


In [0]:
# Query pertanyaan berbasis istilah atau kata kunci spesifik
query_2 = "6G broadband"

# Menjalankan pencarian teks berbasis kata kunci leksikal (Full-Text Search)
results_fulltext = vs_index.similarity_search(
    query_text=query_2,
    columns=["path", "chunk"],
    query_type="FULL_TEXT",
    num_results=3
)

# Mengekstrak list hasil pencarian full-text
passages_fulltext = results_fulltext.get("result", {}).get("data_array", [])

# Menampilkan hasil pencarian kata kunci
display(passages_fulltext)

### Langkah 8: Strategi Retrieval 3 - Pencarian Hibrida (Hybrid Search)
Menggabungkan keunggulan pencarian semantik (vektor) dan pencarian leksikal (full-text) dengan parameter `query_type="hybrid"`. Metode ini memberikan hasil yang optimal dengan menangkap kesesuaian makna sekaligus ketepatan istilah kata kunci.


In [0]:
# Query pertanyaan yang membutuhkan pencocokan kata kunci teknis sekaligus pemahaman konteks semantik
query_3 = "Which products have IP69?"

# Menjalankan pencarian hibrida (gabungan vektor semantik dan pencocokan kata kunci leksikal)
results_hybrid = vs_index.similarity_search(
    query_text=query_3,
    columns=["path", "chunk"],
    query_type="hybrid",
    num_results=5
)

# Mengekstrak list hasil pencarian hibrida
passages_hybrid = results_hybrid.get("result", {}).get("data_array", [])

# Menampilkan hasil pencarian hibrida
display(passages_hybrid)

### Langkah 9: Strategi Retrieval 4 - Pencarian dengan Filter Metadata (Filtered Search)
Menerapkan filter pada metadata dokumen menggunakan parameter `filters`. Pada contoh ini, kita mencari informasi mengenai baterai namun mengecualikan dokumen tertentu (`08_User_Manual.pdf`) dari hasil pencarian.


In [0]:
# Query pertanyaan mengenai prosedur teknis penggantian baterai
query_4 = "How to replace the battery of dPhone D1?"

# Menjalankan pencarian semantik dengan memfilter/mengecualikan file manual tertentu berdasarkan path
results_filtered = vs_index.similarity_search(
    query_text=query_4,
    columns=["path", "chunk"],
    filters={"path NOT" : f"dbfs:/Volumes/workspace_7474652544329313/genai_course/dataset/docs/08_User_Manual.pdf"},
    num_results=3
)

# Mengekstrak list hasil pencarian terfilter
passages_filtered = results_filtered.get("result", {}).get("data_array", [])

# Menampilkan hasil pencarian terfilter
display(passages_filtered)

### Langkah 10: Strategi Re-ranking 1 - Databricks Managed Reranker
Menggunakan `DatabricksReranker` untuk menilai ulang relevansi antara teks query dan potongan dokumen (*cross-encoder scoring*). Re-ranking membantu memastikan dokumen yang paling tepat secara kontekstual berada di urutan teratas.


In [0]:
from databricks.vector_search.reranker import DatabricksReranker

# Query pertanyaan mengenai komponen kamera spesifik
query_5 = "Explain the Halo camera in dPhone?"

# Menjalankan similarity search dengan perangkingan ulang menggunakan Databricks Reranker
results_reranked = vs_index.similarity_search(
    query_text=query_5,
    columns=["path", "chunk"],
    num_results=4,
    reranker=DatabricksReranker(columns_to_rerank=["chunk"])
)

# Mengekstrak list hasil pencarian yang sudah diperingkat ulang
reranked_passages = results_reranked.get("result", {}).get("data_array", [])

# Menampilkan hasil pencarian dengan perangkingan ulang
display(reranked_passages)

### Langkah 11: Strategi Re-ranking 2 - FlashRank Local Cross-Encoder Reranker
Menggunakan pustaka `flashrank` dengan model ringan `rank-T5-flan` yang dieksekusi secara lokal di cluster. Menghasilkan skor relevansi terkalibrasi (*relevance score*) untuk setiap dokumen kandidat tanpa biaya panggilan API eksternal.


In [0]:
from flashrank import Ranker, RerankRequest

# Menentukan direktori cache untuk model FlashRank pada Unity Catalog Volume
cache_dir = f"{course.cache_volume}/flashrank"
# Menginisialisasi model reranker lokal ringan rank-T5-flan
ranker = Ranker(model_name="rank-T5-flan", cache_dir=cache_dir)

# Menyiapkan format dokumen dari hasil retrieval query_1
docs = [
    {"file": doc[0], "text": doc[1]}
    for doc in results.get("result", {}).get("data_array", [])
]

# Mengirim permintaan rerank berdasarkan teks pertanyaan dan daftar dokumen kandidat
rerankrequest = RerankRequest(query=query_1, passages=docs)
passages_flashrank = ranker.rerank(rerankrequest)

# Memformat output hasil reranking dengan menyertakan skor relevansi (score)
formatted_output = [
    {"file": doc["file"], "text": doc["text"], "score": float(doc["score"])} 
    for doc in passages_flashrank
]

# Menampilkan dokumen yang telah diurutkan ulang beserta skor relevansinya
display(formatted_output)